# Phase 4: Feature Engineering & Sparse TF-IDF Representation

**Project:** Indian E-Commerce Recommendation System  
**Objective:** Transform unstructured product titles and verified category metadata into high-dimensional, normalized TF-IDF feature vectors.

### Scalability Highlights:
- Uses **Scikit-learn `TfidfVectorizer`** with sublinear term-frequency scaling and n-gram range `(1, 2)`.
- Represents 76,327 products in a **Compressed Sparse Row (CSR) matrix** of shape `(76327, 25000)`.
- Avoids generating a prohibitive dense $76,327 \times 76,327$ matrix (~46 GB RAM). Sparse matrix requires only ~6.75 MB on disk and ~15 MB in memory.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from scipy import sparse
import joblib

sys.path.append(os.path.abspath(".."))
from src.features.build_features import build_product_feature_text

ARTIFACTS_DIR = os.path.join("..", "artifacts")
PROCESSED_CSV = os.path.join("..", "data", "processed", "products_cleaned.csv")

df = pd.read_csv(PROCESSED_CSV, low_memory=False)
print(f"Loaded {len(df):,} products.")

## 1. Feature Text Construction Example
Examine how raw category and title are combined into a clean, anchored feature document.

In [ ]:
for i in [0, 5000, 15000]:
    sample_row = df.iloc[i]
    feature_doc = build_product_feature_text(sample_row)
    print(f"Product ID: {sample_row['product_id']}")
    print(f"Category:   {sample_row['category']}")
    print(f"Raw Title:  {sample_row['title'][:70]}...")
    print(f"Document:   {feature_doc[:90]}...")
    print("-" * 70)

## 2. Load and Inspect Saved TF-IDF Artifacts

In [ ]:
matrix = sparse.load_npz(os.path.join(ARTIFACTS_DIR, "tfidf_matrix.npz"))
vectorizer = joblib.load(os.path.join(ARTIFACTS_DIR, "tfidf_vectorizer.joblib"))
index_maps = joblib.load(os.path.join(ARTIFACTS_DIR, "product_index_map.joblib"))

print(f"Sparse Matrix Shape: {matrix.shape}")
print(f"Number of non-zero entries (nnz): {matrix.nnz:,}")
print(f"Sparsity: {100.0 * (1.0 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])):.2f}%")
print(f"Indexed products: {len(index_maps['id_to_idx']):,}")

## 3. Vocabulary Exploration
Sample important n-grams captured by the vectorizer.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
print(f"Total Vocabulary Size: {len(feature_names):,}")
print("Sample Unigrams and Bigrams:")
np.random.seed(42)
sample_terms = np.random.choice(feature_names, 25, replace=False)
for t in sample_terms:
    print(f" - {t}")